In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
db_path = '/content/drive/MyDrive/UPN VY/Bootcamp/LnT Binus Machine Learning/Final Project/superstore.sqlite'
conn = sqlite3.connect(db_path)

Mounted at /content/drive


# EDA

In [ ]:
query = """
SELECT
    o.order_key,
    o.order_id_raw,
    o.order_date,
    o.ship_date,
    o.ship_mode,
    o.order_priority,

    oi.row_id,
    oi.discount,
    oi.quantity,
    oi.sales,
    oi.profit,
    oi.shipping_cost,
    oi.year,
    oi.week_num,

    c.customer_id,
    c.customer_name,
    c.segment,

    l.location_id,
    l.city,
    l.state,
    l.country,
    l.region,
    l.market,
    l.market2,

    p.product_id,
    p.category,
    p.sub_category,
    p.product_name,

    v.raw_product_name

FROM order_items oi
INNER JOIN orders o        ON oi.order_key = o.order_key
INNER JOIN dim_customers c ON o.customer_id = c.customer_id
INNER JOIN dim_locations l ON o.location_id = l.location_id
INNER JOIN dim_products p  ON oi.product_id = p.product_id
LEFT JOIN dim_product_name_variants v ON p.product_id = v.product_id;
"""

df = pd.read_sql_query(query, conn)
conn.close()
print("Data berhasil diload. Ukuran awal:", df.shape)

Data berhasil diload. Ukuran awal: (55996, 29)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55996 entries, 0 to 55995
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_key         55996 non-null  object 
 1   order_id_raw      55996 non-null  object 
 2   order_date        55996 non-null  object 
 3   ship_date         55996 non-null  object 
 4   ship_mode         55996 non-null  object 
 5   order_priority    55996 non-null  object 
 6   row_id            55996 non-null  int64  
 7   discount          55996 non-null  float64
 8   quantity          55996 non-null  int64  
 9   sales             55996 non-null  float64
 10  profit            55996 non-null  float64
 11  shipping_cost     55996 non-null  float64
 12  year              55996 non-null  int64  
 13  week_num          55996 non-null  int64  
 14  customer_id       55996 non-null  object 
 15  customer_name     55996 non-null  object 
 16  segment           55996 non-null  object

In [ ]:
print("total rows:", len(df))
print("unique row_id:", df['row_id'].nunique())
print("unique product_id:", df['product_id'].nunique())
print("rows per product_id (top 10 by count):")
print(df.groupby('product_id').size().sort_values(ascending=False).head(10))

total rows: 55996
unique row_id: 51290
unique product_id: 10292
rows per product_id (top 10 by count):
product_id
OFF-AR-10003651    105
OFF-AR-10003829     93
OFF-BI-10002799     90
OFF-BI-10003708     90
OFF-BI-10004632     72
OFF-BI-10004654     66
OFF-AR-10000833     57
FUR-CH-10003354     56
OFF-BI-10002570     54
OFF-BI-10004140     50
dtype: int64


In [ ]:
# Cari row_id mana saja yang punya lebih dari 1 baris (karena multi-variant)
dup_mask = df['row_id'].duplicated(keep=False)
print("rows affected by fan-out:", dup_mask.sum())

rows affected by fan-out: 9098


In [ ]:
# Lihat contoh konkretnya
df[dup_mask].sort_values('row_id').head(10)[
    ['row_id', 'product_id', 'raw_product_name', 'product_name', 'sales', 'profit']
]

,row_id,product_id,raw_product_name,product_name,sales,profit
15,16,OFF-FA-10000038,"Advantus Clamps, Metal","Stockwell Clamps, Assorted Sizes",13.0,1.020
16,16,OFF-FA-10000038,"Stockwell Clamps, Assorted Sizes","Stockwell Clamps, Assorted Sizes",13.0,1.020
27,27,OFF-ST-10004296,"Rogers Box, Single Width","Smead Folders, Industrial",80.0,34.400
28,27,OFF-ST-10004296,"Smead Folders, Industrial","Smead Folders, Industrial",80.0,34.400
32,30,TEC-PH-10002483,"Motorola Office Telephone, with Caller ID","Motorola Office Telephone, with Caller ID",115.0,-49.824
31,30,TEC-PH-10002483,Motorola Moto X,"Motorola Office Telephone, with Caller ID",115.0,-49.824
51,49,FUR-FU-10003608,"Advantus Light Bulb, Black","Advantus Light Bulb, Black",27.0,7.480
52,49,FUR-FU-10003608,"Tenex Light Bulb, Erganomic","Advantus Light Bulb, Black",27.0,7.480
54,51,OFF-SU-10003665,"Kleencut Trimmer, Serrated","Stiletto Letter Opener, Steel",67.0,28.720
55,51,OFF-SU-10003665,"Stiletto Letter Opener, Steel","Stiletto Letter Opener, Steel",67.0,28.720


In [ ]:
df_clean = df.drop_duplicates(subset='row_id').copy()
print("before:", len(df), "after dedup:", len(df_clean))

before: 55996 after dedup: 51290


In [ ]:
# BUG FIX: describe() harus pakai df_clean (sudah dedup), bukan df mentah (masih ada fan-out)
df_clean.describe()

,row_id,discount,quantity,sales,profit,shipping_cost,year,week_num,location_id
count,51290.00000,51290.000000,51290.000000,51290.000000,51290.000000,51290.000000,51290.000000,51290.000000,51290.000000
mean,25645.50000,0.142908,3.476545,246.498440,28.610982,26.375818,2012.777208,31.287112,1712.295652
std,14806.29199,0.212280,2.278766,487.567175,174.340972,57.296810,1.098931,14.429795,1275.144101
min,1.00000,0.000000,1.000000,0.000000,-6599.978000,0.002000,2011.000000,1.000000,1.000000
25%,12823.25000,0.000000,2.000000,31.000000,0.000000,2.610000,2012.000000,20.000000,540.000000
50%,25645.50000,0.000000,3.000000,85.000000,9.240000,7.790000,2013.000000,33.000000,1330.000000
75%,38467.75000,0.200000,5.000000,251.000000,36.810000,24.450000,2014.000000,44.000000,3124.000000
max,51290.00000,0.850000,14.000000,22638.000000,8399.976000,933.570000,2014.000000,53.000000,3819.000000


In [ ]:
# cek rentang waktu
print(df_clean['order_date'].min(), "-", df_clean['order_date'].max())
print(df_clean.groupby('year').size())

2011-01-01 00:00:00 - 2014-12-31 00:00:00
year
2011     8998
2012    10962
2013    13799
2014    17531
dtype: int64


In [ ]:
# cek distribusi profit
print("negative profit rows:", (df_clean['profit'] < 0).sum(),
      f"({(df_clean['profit'] < 0).mean():.1%})")
print(df_clean['profit'].describe())

negative profit rows: 12544 (24.5%)
count    51290.000000
mean        28.610982
std        174.340972
min      -6599.978000
25%          0.000000
50%          9.240000
75%         36.810000
max       8399.976000
Name: profit, dtype: float64


In [ ]:
# cek profitabilitas per kategori
cat_summary = df_clean.groupby('category').agg(
    total_sales=('sales', 'sum'),
    total_profit=('profit', 'sum'),
    median_profit=('profit', 'median'),
    pct_loss=('profit', lambda x: (x < 0).mean()),
    n_transactions=('row_id', 'count')
).sort_values('total_profit', ascending=False)
print(cat_summary)

                 total_sales  total_profit  median_profit  pct_loss  \
category                                                              
Technology         4744691.0  663778.73318        29.9400  0.239030   
Office Supplies    3787330.0  518473.83430         6.5538  0.223931   
Furniture          4110884.0  285204.72380        15.5022  0.315614   

                 n_transactions  
category                         
Technology                10141  
Office Supplies           31273  
Furniture                  9876  


In [ ]:
# cek hierarki market vs region
print(df_clean.groupby(['market', 'region']).size())

market  region        
APAC    Central Asia      2048
        North Asia        2338
        Oceania           3487
        Southeast Asia    3129
Africa  Africa            4587
Canada  Canada             384
EMEA    EMEA              5029
EU      Central           5822
        North             2141
        South             2037
LATAM   Caribbean         1690
        Central           2972
        North             2644
        South             2988
US      Central           2323
        East              2848
        South             1620
        West              3203
dtype: int64


In [ ]:
# cek apakah outlier profit terkonsentrasi di sub_category tertentu
print(df_clean.nsmallest(15, 'profit')[['sub_category', 'discount', 'sales', 'profit']])
print(df_clean.nlargest(15, 'profit')[['sub_category', 'discount', 'sales', 'profit']])

      sub_category  discount   sales     profit
43464     Machines      0.70  4500.0 -6599.9780
53357       Phones      0.60  3085.0 -4088.3760
35480     Machines      0.50  8000.0 -3839.9904
45713      Binders      0.80  2178.0 -3701.8928
38103     Machines      0.70  2550.0 -3399.9800
11603   Appliances      0.50  3400.0 -3059.8200
32742       Phones      0.50  3499.0 -3009.4350
40358      Binders      0.80  1890.0 -2929.4845
50900       Tables      0.70  2171.0 -2750.2800
38267     Machines      0.70  1800.0 -2639.9912
29623       Tables      0.80  1287.0 -2380.3500
40706      Binders      0.80  1525.0 -2287.7820
13054       Chairs      0.50  2571.0 -2211.1650
16745    Bookcases      0.50  2830.0 -1981.3950
12987       Tables      0.85   555.0 -1924.5420
      sub_category  discount    sales     profit
42394      Copiers       0.0  17500.0  8399.9760
43884      Copiers       0.0  14000.0  6719.9808
39455      Copiers       0.0  10500.0  5039.9856
44882      Binders       0.0   9893.

In [ ]:
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])
monthly = df_clean.groupby(df_clean['order_date'].dt.to_period('M')).size()
print(monthly)

order_date
2011-01     433
2011-02     378
2011-03     539
2011-04     564
2011-05     566
2011-06     917
2011-07     495
2011-08     878
2011-09    1052
2011-10     774
2011-11    1139
2011-12    1263
2012-01     539
2012-02     429
2012-03     662
2012-04     657
2012-05     840
2012-06    1136
2012-07     659
2012-08    1083
2012-09    1223
2012-10     990
2012-11    1460
2012-12    1284
2013-01     709
2013-02     605
2013-03     766
2013-04     785
2013-05    1057
2013-06    1530
2013-07     925
2013-08    1442
2013-09    1682
2013-10    1100
2013-11    1596
2013-12    1602
2014-01     918
2014-02     756
2014-03    1068
2014-04    1052
2014-05    1284
2014-06    1748
2014-07    1087
2014-08    1674
2014-09    2018
2014-10    1626
2014-11    2147
2014-12    2153
Freq: M, dtype: int64


In [ ]:
dec_2014 = df_clean[(df_clean['order_date'].dt.year == 2014) & (df_clean['order_date'].dt.month == 12)]
print(dec_2014['order_date'].dt.day.value_counts().sort_index())
print("last order date:", dec_2014['order_date'].max())

order_date
1      74
2      94
3     101
4      96
5      78
6      40
7       7
8      58
9      96
10    114
11     85
12     85
13     50
14     10
15     82
16     68
17     76
18     69
19    101
20     29
21      6
22     46
23     99
24     76
25     79
26    110
27     54
28     13
29    116
30     79
31     62
Name: count, dtype: int64
last order date: 2014-12-31 00:00:00


In [ ]:
# cek apakah Technology benar2 punya spread lebih ekstrem dari Furniture
print(df_clean.groupby('category')['profit'].agg(['std', 'min', 'max']))

                        std        min       max
category                                        
Furniture        205.759936 -2750.2800  2316.510
Office Supplies  109.749251 -3701.8928  4946.370
Technology       271.215376 -6599.9780  8399.976


In [ ]:
# cek hubungan discount vs profit langsung, per kategori
print(df_clean.groupby('category').apply(
    lambda g: g['discount'].corr(g['profit'])
))

category
Furniture         -0.492226
Office Supplies   -0.253860
Technology        -0.376786
dtype: float64


/tmp/ipykernel_772/4118690467.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df_clean.groupby('category').apply(


# Regression Modeling

In [ ]:
df_clean = df.drop_duplicates(subset='row_id').copy()
assert df_clean['row_id'].is_unique, "masih ada duplikasi row_id, cek ulang"

In [ ]:
target_col = 'profit'

categorical_cols = [
    'sub_category',   # bukan 'category' — heterogenitas terbukti signifikan
    'region',
    'market',         # market2 didrop — verifikasi crosstab menunjukkan market2 cuma versi collapse dari market (Canada+US -> "North America"), tidak ada informasi baru
    'segment',
    'ship_mode',
    'order_priority',
]

numeric_cols = [
    'discount',
    'quantity',
    'sales',           # dimasukkan — lihat justifikasi empiris di markdown cell sebelum ini
    'shipping_cost',   # dimasukkan — use case dikonfirmasi post-fulfillment (data transaksi lengkap tersedia saat prediksi)
]

id_cols_for_later = [
    'product_id',     # cardinality tinggi (10292 unique) -- BUKAN untuk one-hot langsung, disiapkan untuk historical
    'customer_id',    # aggregate encoding di iterasi berikutnya (di luar scope notebook ini yang berhenti di baseline)
]

date_col = 'order_date'  # dipakai untuk derive month/quarter/day_of_week

# Kolom yang eksplisit di-exclude, didokumentasikan di sini
excluded_cols = [
    'row_id', 'order_key', 'order_id_raw',
    'customer_name', 'product_name', 'raw_product_name',
    'market2',
]


In [ ]:
selected_cols = [target_col] + categorical_cols + numeric_cols + id_cols_for_later + [date_col]

df_reg = df_clean[selected_cols].copy()

In [ ]:
# sanity check
print("shape df_reg:", df_reg.shape)
print("\nmissing values:\n", df_reg.isna().sum())
print("\ndtypes:\n", df_reg.dtypes)
print("\nsample:\n", df_reg.head())

shape df_reg: (51290, 14)

missing values:
 profit            0
sub_category      0
region            0
market            0
segment           0
ship_mode         0
order_priority    0
discount          0
quantity          0
sales             0
shipping_cost     0
product_id        0
customer_id       0
order_date        0
dtype: int64

dtypes:
 profit            float64
sub_category       object
region             object
market             object
segment            object
ship_mode          object
order_priority     object
discount          float64
quantity            int64
sales             float64
shipping_cost     float64
product_id         object
customer_id        object
order_date         object
dtype: object

sample:
    profit sub_category region market   segment       ship_mode order_priority  \
0    4.56       Labels  North  LATAM  Consumer  Standard Class         Medium   
1   90.72  Furnishings  South  LATAM  Consumer  Standard Class         Medium   
2   54.08    Bookcases

In [ ]:
# date extraction
df_reg['order_date'] = pd.to_datetime(df_reg['order_date'])

df_reg['order_month'] = df_reg['order_date'].dt.month
df_reg['order_quarter'] = df_reg['order_date'].dt.quarter
df_reg['order_dayofweek'] = df_reg['order_date'].dt.dayofweek

In [ ]:
# sanity check
print(df_reg[['order_date', 'order_month', 'order_quarter', 'order_dayofweek']].head())
print("\nrange tahun:", df_reg['order_date'].dt.year.min(), "-", df_reg['order_date'].dt.year.max())

  order_date  order_month  order_quarter  order_dayofweek
0 2014-10-02           10              4                3
1 2012-10-15           10              4                0
2 2012-10-15           10              4                0
3 2012-10-15           10              4                0
4 2012-10-15           10              4                0

range tahun: 2011 - 2014


Baseline Model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
baseline_cols_categorical = [
    'sub_category', 'region', 'market', 'segment', 'ship_mode', 'order_priority'
]
baseline_cols_numeric = [
    'discount', 'quantity', 'sales', 'shipping_cost', 'order_month', 'order_quarter', 'order_dayofweek'
]

df_baseline = df_reg[['order_date', 'profit'] + baseline_cols_categorical + baseline_cols_numeric].copy()

print("shape:", df_baseline.shape)
print("missing:\n", df_baseline.isna().sum())


shape: (51290, 15)
missing:
 order_date         0
profit             0
sub_category       0
region             0
market             0
segment            0
ship_mode          0
order_priority     0
discount           0
quantity           0
sales              0
shipping_cost      0
order_month        0
order_quarter      0
order_dayofweek    0
dtype: int64


In [ ]:
# temporal split
df_baseline = df_baseline.sort_values('order_date').reset_index(drop=True)
cutoff_date = pd.Timestamp('2014-01-01')

train_mask = df_baseline['order_date'] < cutoff_date
df_train = df_baseline[train_mask].drop(columns='order_date')
df_test = df_baseline[~train_mask].drop(columns='order_date')

X_train = df_train.drop(columns='profit')
y_train = df_train['profit']
X_test = df_test.drop(columns='profit')
y_test = df_test['profit']

print(f"\ntrain: {X_train.shape[0]} rows, test: {X_test.shape[0]} rows")


train: 33759 rows, test: 17531 rows


In [ ]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), baseline_cols_categorical),
    ],
    remainder='passthrough'  # numeric_cols lolos apa adanya
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=5,   # sedikit regularisasi, cegah overfit ke outlier ekstrem
        n_jobs=-1,
        random_state=42
    ))
])

In [ ]:
# Fit and evaluate model
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\nMAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2:   {r2:.4f}")


MAE:  34.790
RMSE: 94.370
R2:   0.7068


In [ ]:
# feature importance
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
importances = model.named_steps['regressor'].feature_importances_

fi = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values(
    'importance', ascending=False
)
print("\ntop 15 feature importance:")
print(fi.head(15))


top 15 feature importance:
                          feature  importance
50               remainder__sales    0.556684
48            remainder__discount    0.323339
51       remainder__shipping_cost    0.030463
49            remainder__quantity    0.015842
52         remainder__order_month    0.013132
54     remainder__order_dayofweek    0.010052
30               cat__market_APAC    0.004069
20            cat__region_Central    0.003501
53       remainder__order_quarter    0.003322
47     cat__order_priority_Medium    0.003314
37          cat__segment_Consumer    0.003072
43  cat__ship_mode_Standard Class    0.002459
38         cat__segment_Corporate    0.002231
34                 cat__market_EU    0.002060
45       cat__order_priority_High    0.001917


# Feature Engineering Lanjutan + XGBoost

### Interaction / ratio terms

Dua fitur turunan yang ditambahkan, keduanya row-wise (tidak butuh histori/aggregate lain, jadi tidak ada risiko leakage temporal):

- `discount_x_quantity` — menangkap efek gabungan diskon besar pada order berkuantitas besar (bulk order didiskon berat), pola yang sudah terlihat di eksplorasi awal (Tables, Binders).
- `sales_per_unit = sales / quantity` — proxy unit price. Karena `sales` sudah termasuk fitur, ini sebagian informasinya tumpang tindih, tapi tree-based model melakukan split axis-aligned dan tidak otomatis menemukan rasio antar dua kolom — jadi tetap berpotensi membantu split yang lebih tajam.

In [ ]:
df_reg['discount_x_quantity'] = df_reg['discount'] * df_reg['quantity']
df_reg['sales_per_unit'] = df_reg['sales'] / df_reg['quantity']

engineered_numeric_cols = ['discount_x_quantity', 'sales_per_unit']
print(df_reg[['discount', 'quantity', 'discount_x_quantity', 'sales', 'sales_per_unit']].head())

   discount  quantity  discount_x_quantity  sales  sales_per_unit
0       0.0         3                  0.0   13.0        4.333333
1       0.0         8                  0.0  252.0       31.500000
2       0.0         2                  0.0  193.0       96.500000
3       0.0         4                  0.0   35.0        8.750000
4       0.0         2                  0.0   72.0       36.000000


In [ ]:
baseline_cols_categorical = [
    'sub_category', 'region', 'market', 'segment', 'ship_mode', 'order_priority'
]
numeric_cols_v2 = [
    'discount', 'quantity', 'sales', 'shipping_cost',
    'discount_x_quantity', 'sales_per_unit',
    'order_month', 'order_quarter', 'order_dayofweek'
]

df_v2 = df_reg[['order_date', 'profit', 'product_id', 'customer_id']
               + baseline_cols_categorical + numeric_cols_v2].copy()
df_v2 = df_v2.sort_values('order_date').reset_index(drop=True)

cutoff_date = pd.Timestamp('2014-01-01')
train_mask = df_v2['order_date'] < cutoff_date
df_train_v2 = df_v2[train_mask].drop(columns='order_date').copy()
df_test_v2  = df_v2[~train_mask].drop(columns='order_date').copy()

print(f"train: {df_train_v2.shape[0]} rows, test: {df_test_v2.shape[0]} rows")

train: 33759 rows, test: 17531 rows


In [ ]:
def fit_smoothed_target_encoding(train_df, col, target_col, m=10):
    """Fit HANYA dari train_df. m = kekuatan shrinkage ke rata-rata global."""
    global_mean = train_df[target_col].mean()
    agg = train_df.groupby(col)[target_col].agg(['mean', 'count'])
    agg['smoothed'] = (agg['count'] * agg['mean'] + m * global_mean) / (agg['count'] + m)
    return agg['smoothed'].to_dict(), global_mean

numeric_cols_v2_te = list(numeric_cols_v2)
for col in ['product_id', 'customer_id']:
    mapping, global_mean = fit_smoothed_target_encoding(df_train_v2, col, 'profit', m=10)
    enc_col = f'{col}_te'
    df_train_v2[enc_col] = df_train_v2[col].map(mapping)
    # kategori yang tidak muncul di train (unseen product/customer di test) -> fallback ke rata-rata global
    df_test_v2[enc_col] = df_test_v2[col].map(mapping).fillna(global_mean)
    numeric_cols_v2_te.append(enc_col)

df_train_v2 = df_train_v2.drop(columns=['product_id', 'customer_id'])
df_test_v2  = df_test_v2.drop(columns=['product_id', 'customer_id'])

X_train_v2 = df_train_v2.drop(columns='profit'); y_train_v2 = df_train_v2['profit']
X_test_v2  = df_test_v2.drop(columns='profit');  y_test_v2  = df_test_v2['profit']

### Fit XGBoost — dengan vs tanpa target encoding (ablation)

In [ ]:
from xgboost import XGBRegressor

preprocessor_v2 = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), baseline_cols_categorical)],
    remainder='passthrough'
)

xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor_v2),
    ('regressor', XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1
    ))
])

xgb_model.fit(X_train_v2, y_train_v2)
pred_with_te = xgb_model.predict(X_test_v2)

print("XGB + target encoding:")
print(f"  MAE:  {mean_absolute_error(y_test_v2, pred_with_te):.3f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test_v2, pred_with_te)):.3f}")
print(f"  R2:   {r2_score(y_test_v2, pred_with_te):.4f}")

XGB + target encoding:
  MAE:  30.123
  RMSE: 106.448
  R2:   0.6269


In [ ]:
# Ablation: XGBoost TANPA target encoding, cuma interaction terms
drop_te_cols = ['product_id_te', 'customer_id_te']
X_train_noTE = X_train_v2.drop(columns=drop_te_cols)
X_test_noTE  = X_test_v2.drop(columns=drop_te_cols)

preprocessor_noTE = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), baseline_cols_categorical)],
    remainder='passthrough'
)
xgb_model_noTE = Pipeline(steps=[
    ('preprocessor', preprocessor_noTE),
    ('regressor', XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1
    ))
])
xgb_model_noTE.fit(X_train_noTE, y_train_v2)
pred_no_te = xgb_model_noTE.predict(X_test_noTE)

print("XGB tanpa target encoding (cuma interaction terms):")
print(f"  MAE:  {mean_absolute_error(y_test_v2, pred_no_te):.3f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test_v2, pred_no_te)):.3f}")
print(f"  R2:   {r2_score(y_test_v2, pred_no_te):.4f}")

XGB tanpa target encoding (cuma interaction terms):
  MAE:  34.714
  RMSE: 95.267
  R2:   0.7012


In [ ]:
feature_names_v2 = xgb_model_noTE.named_steps['preprocessor'].get_feature_names_out()
importances_v2 = xgb_model_noTE.named_steps['regressor'].feature_importances_

fi_v2 = pd.DataFrame({'feature': feature_names_v2, 'importance': importances_v2}).sort_values(
    'importance', ascending=False
)
print("top 15 feature importance (XGBoost, tanpa target encoding):")
print(fi_v2.head(15))

top 15 feature importance (XGBoost, tanpa target encoding):
                           feature  importance
48             remainder__discount    0.116680
50                remainder__sales    0.083869
52  remainder__discount_x_quantity    0.074809
3        cat__sub_category_Binders    0.063430
6        cat__sub_category_Copiers    0.045497
36                  cat__market_US    0.031838
15      cat__sub_category_Supplies    0.031521
29                cat__region_West    0.031116
53       remainder__sales_per_unit    0.028680
47      cat__order_priority_Medium    0.028340
20             cat__region_Central    0.025995
49             remainder__quantity    0.025769
51        remainder__shipping_cost    0.025691
46         cat__order_priority_Low    0.019594
56      remainder__order_dayofweek    0.017923


# Klasifikasi: Prediksi Keterlambatan Pengiriman

### Definisi target

Satu baris data adalah satu order (bukan satu item), karena pengiriman terjadi per order. Order dianggap terlambat (`is_late = 1`) jika selisih `ship_date` dan `order_date` melebihi batas hari untuk `ship_mode`-nya: Same Day 0, First Class 2, Second Class 3, Standard Class 5.

Dataset tidak punya tanggal janji pengiriman, jadi batas ini adalah asumsi yang saya ambil dari median lama kirim tiap mode. Kelas positif adalah terlambat, sehingga Precision, Recall, dan F1 dihitung untuk kelas itu.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from xgboost import XGBClassifier

# batas hari per ship_mode (asumsi: median lama kirim tiap mode)
late_days_limit = {'Same Day': 0, 'First Class': 2, 'Second Class': 3, 'Standard Class': 5}

orders_src = df_clean.copy()
orders_src['order_date'] = pd.to_datetime(orders_src['order_date'])
orders_src['ship_date']  = pd.to_datetime(orders_src['ship_date'])

# agregasi item -> order
df_ord = orders_src.groupby('order_key').agg(
    order_date=('order_date', 'first'),
    ship_date=('ship_date', 'first'),
    ship_mode=('ship_mode', 'first'),
    order_priority=('order_priority', 'first'),
    segment=('segment', 'first'),
    region=('region', 'first'),
    market=('market', 'first'),
    n_items=('row_id', 'count'),
    total_quantity=('quantity', 'sum'),
    total_sales=('sales', 'sum'),
    avg_discount=('discount', 'mean'),
    total_shipping_cost=('shipping_cost', 'sum'),
    n_sub_categories=('sub_category', 'nunique'),
).reset_index()

# target dibentuk dari ship_date, lalu ship_date tidak dipakai lagi
df_ord['ship_days'] = (df_ord['ship_date'] - df_ord['order_date']).dt.days
df_ord['is_late'] = (df_ord['ship_days'] > df_ord['ship_mode'].map(late_days_limit)).astype(int)

df_ord['order_month'] = df_ord['order_date'].dt.month
df_ord['order_quarter'] = df_ord['order_date'].dt.quarter
df_ord['order_dayofweek'] = df_ord['order_date'].dt.dayofweek
df_ord = df_ord.sort_values('order_date').reset_index(drop=True)

print("jumlah order:", len(df_ord))
print(df_ord['is_late'].value_counts(normalize=True).round(3))
print(df_ord.groupby(df_ord['order_date'].dt.year)['is_late'].mean().round(3))
print(df_ord.groupby('ship_mode')['is_late'].mean().round(3))

### Split temporal dan one-hot encoding

`ship_date` dan `ship_days` hanya dipakai untuk membentuk target dan tidak masuk fitur. Split tetap temporal (train 2011-2013, test 2014), dan encoder di-fit hanya dari data train.

In [ ]:
clf_cat_cols = ['ship_mode', 'order_priority', 'segment', 'region', 'market']
clf_num_cols = ['n_items', 'total_quantity', 'total_sales', 'avg_discount',
                'total_shipping_cost', 'n_sub_categories',
                'order_month', 'order_quarter', 'order_dayofweek']

cutoff_date = pd.Timestamp('2014-01-01')
clf_train = df_ord[df_ord['order_date'] < cutoff_date]
clf_test  = df_ord[df_ord['order_date'] >= cutoff_date]
y_train_c, y_test_c = clf_train['is_late'].values, clf_test['is_late'].values

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(clf_train[clf_cat_cols])
X_train_c = np.hstack([ohe.transform(clf_train[clf_cat_cols]), clf_train[clf_num_cols].values])
X_test_c  = np.hstack([ohe.transform(clf_test[clf_cat_cols]),  clf_test[clf_num_cols].values])
clf_feature_names = list(ohe.get_feature_names_out(clf_cat_cols)) + clf_num_cols

print(f"train: {X_train_c.shape}, test: {X_test_c.shape}")
print(f"late rate train: {y_train_c.mean():.3f}, test: {y_test_c.mean():.3f}")

### Baseline naif

Dua pembanding sederhana yang harus dikalahkan model: selalu menebak tepat waktu, dan aturan satu baris yang menebak terlambat untuk order berprioritas Low.

In [ ]:
results = []

def evaluate(name, y_true, y_pred, y_score):
    row = {
        'model': name,
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred),
        'f1':        f1_score(y_true, y_pred),
        'roc_auc':   roc_auc_score(y_true, y_score),
    }
    results.append(row)
    print({k: (round(v, 4) if k != 'model' else v) for k, v in row.items()})

# 1) selalu tebak 'tepat waktu'
evaluate('majority (selalu 0)', y_test_c, np.zeros_like(y_test_c), np.zeros(len(y_test_c)))

# 2) aturan satu baris: prioritas Low -> terlambat
rule_pred = (clf_test['order_priority'] == 'Low').astype(int).values
evaluate('rule: priority == Low', y_test_c, rule_pred, rule_pred)

### Model: Logistic Regression dan XGBoost

Scaling dipakai untuk Logistic Regression karena modelnya sensitif terhadap skala fitur. XGBoost tidak memerlukannya.

In [ ]:
# Logistic Regression (scaled)
scaler = StandardScaler().fit(X_train_c)
lr_clf = LogisticRegression(max_iter=2000).fit(scaler.transform(X_train_c), y_train_c)
evaluate('LogisticRegression', y_test_c,
         lr_clf.predict(scaler.transform(X_test_c)),
         lr_clf.predict_proba(scaler.transform(X_test_c))[:, 1])

# XGBoost
xgb_params = dict(n_estimators=300, max_depth=5, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_clf = XGBClassifier(**xgb_params).fit(X_train_c, y_train_c)
evaluate('XGBoost', y_test_c, xgb_clf.predict(X_test_c), xgb_clf.predict_proba(X_test_c)[:, 1])

# XGBoost dengan pembobotan kelas terlambat
spw = (y_train_c == 0).sum() / (y_train_c == 1).sum()
xgb_clf_w = XGBClassifier(scale_pos_weight=spw, **xgb_params).fit(X_train_c, y_train_c)
evaluate('XGBoost + scale_pos_weight', y_test_c, xgb_clf_w.predict(X_test_c),
         xgb_clf_w.predict_proba(X_test_c)[:, 1])

In [ ]:
pd.DataFrame(results).round(4)

In [ ]:
fi_clf = pd.DataFrame({'feature': clf_feature_names,
                       'importance': xgb_clf.feature_importances_}).sort_values('importance', ascending=False)
print(fi_clf.head(10))

In [ ]:
# ==========================================
# 1. SIMULASI INPUT USER (DARI WEB FORM NANTINYA)
# ==========================================
# Bayangkan user mengetik angka dan memilih dropdown di web,
# lalu datanya masuk ke sistem dalam bentuk seperti ini:
user_input = {
    "order_date": "2024-01-15",
    "discount": 0.1,
    "quantity": 3,
    "sales": 500,
    "shipping_cost": 20,
    "sub_category": "Binders",
    "region": "Central",
    "market": "US",
    "segment": "Consumer",
    "ship_mode": "Second Class",
    "order_priority": "High"
}

df_input = pd.DataFrame([user_input])
print("1. Data Input User:")
display(df_input)

# ==========================================
# 2. FEATURE ENGINEERING (Sama seperti training)
# ==========================================
# Convert order_date to datetime and extract features
df_input['order_date'] = pd.to_datetime(df_input['order_date'])
df_input['order_month'] = df_input['order_date'].dt.month
df_input['order_quarter'] = df_input['order_date'].dt.quarter
df_input['order_dayofweek'] = df_input['order_date'].dt.dayofweek
df_input = df_input.drop(columns=['order_date']) # Drop the original date column

# Corrected engineered features calculation (should use df_input's own columns)
df_input['discount_x_quantity'] = df_input['discount'] * df_input['quantity']
df_input['sales_per_unit'] = df_input['sales'] / df_input['quantity']

# ==========================================
# 3. PREPARING INPUT FOR MODEL
# ==========================================
# The xgb_model_noTE pipeline expects raw features (categorical as objects/strings, numeric as numbers).
# It will handle one-hot encoding and remainder passthrough internally.

# The features that the preprocessor_noTE expects as input are:
# baseline_cols_categorical (for OHE) and numeric_cols_v2 (for passthrough).
# Combine these lists to get the ordered columns expected by the model's preprocessor.
# Ensure df_input has these columns in the correct order.
model_expected_input_cols = baseline_cols_categorical + numeric_cols_v2
df_model_input = df_input[model_expected_input_cols]

# ==========================================
# 4. PREDIKSI HASIL (Inference)
# ==========================================
# Pass the prepared DataFrame directly to the pipeline.
# The pipeline's preprocessor will handle OHE and passing through numeric features.
prediksi_profit = xgb_model_noTE.predict(df_model_input)

print("\n==========================================")
print(f"💰 ESTIMASI PROFIT: ${prediksi_profit[0]:.2f}")
print("==========================================")

# Logika bisnis tambahan untuk UI (Peringatan Untung/Rugi)
if prediksi_profit[0] < 0:
    print("PERINGATAN: Transaksi ini diprediksi merugikan perusahaan! Pertimbangkan untuk menurunkan diskon.")
else:
    print("STATUS AMAN: Transaksi ini diprediksi menghasilkan keuntungan.")

1. Data Input User:


,order_date,discount,quantity,sales,shipping_cost,sub_category,region,market,segment,ship_mode,order_priority
0,2024-01-15,0.1,3,500,20,Binders,Central,US,Consumer,Second Class,High



💰 ESTIMASI PROFIT: $147.55
STATUS AMAN: Transaksi ini diprediksi menghasilkan keuntungan.


# Download Model

In [ ]:
import joblib, sklearn, xgboost
from google.colab import files

versions = {'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__}

# ---------- 1) Klasifikasi: model + encoder + metadata dalam satu file ----------
clf_artifact = {
    'model': xgb_clf,
    'encoder': ohe,
    'cat_cols': clf_cat_cols,
    'num_cols': clf_num_cols,
    'threshold': 0.5,
    'late_days_limit': late_days_limit,
    'target': 'is_late (1 = ship_days > batas hari per ship_mode)',
    'versions': versions,
}
joblib.dump(clf_artifact, 'xgb_late_classifier.pkl')

# ---------- 2) Regresi: Pipeline sudah termasuk preprocessing ----------
reg_artifact = {
    'model': xgb_model_noTE,
    'feature_cols': list(X_train_noTE.columns),
    'target': 'profit',
    'versions': versions,
}
joblib.dump(reg_artifact, 'xgb_profit_regressor.pkl')

# ---------- 3) Sanity check: reload lalu bandingkan prediksi ----------
loaded = joblib.load('xgb_late_classifier.pkl')
X_check = np.hstack([loaded['encoder'].transform(clf_test[loaded['cat_cols']]),
                     clf_test[loaded['num_cols']].values])
p_loaded = loaded['model'].predict_proba(X_check)[:, 1]
p_orig = xgb_clf.predict_proba(X_test_c)[:, 1]
print("prediksi identik:", np.allclose(p_loaded, p_orig))

loaded_reg = joblib.load('xgb_profit_regressor.pkl')
print("regresi identik:", np.allclose(
    loaded_reg['model'].predict(X_test_noTE[loaded_reg['feature_cols']]),
    xgb_model_noTE.predict(X_test_noTE)))

# ---------- 4) Download (Colab) ----------
files.download('xgb_late_classifier.pkl')
files.download('xgb_profit_regressor.pkl')

In [ ]:
import joblib
import sklearn
import xgboost

versions = {
    "sklearn": sklearn.__version__,
    "xgboost": xgboost.__version__
}

# ============================================================
# CLASSIFIER
# ============================================================

# XGBoost native model
xgb_clf.save_model("xgb_late_classifier.json")

# Komponen sklearn + metadata
clf_meta = {
    "encoder": ohe,
    "cat_cols": clf_cat_cols,
    "num_cols": clf_num_cols,
    "threshold": 0.5,
    "late_days_limit": late_days_limit,
    "target": "is_late (1 = ship_days > batas hari per ship_mode)",
    "versions": versions,
}

joblib.dump(
    clf_meta,
    "xgb_late_classifier_meta.pkl"
)


# ============================================================
# REGRESSOR
# ============================================================

# Ambil preprocessing dan XGBRegressor dari Pipeline
reg_preprocessor = xgb_model_noTE.named_steps["preprocessor"]
reg_model = xgb_model_noTE.named_steps["regressor"]

# XGBoost native model
reg_model.save_model("xgb_profit_regressor.json")

# Preprocessor + metadata
reg_meta = {
    "preprocessor": reg_preprocessor,
    "feature_cols": list(X_train_noTE.columns),
    "target": "profit",
    "versions": versions,
}

joblib.dump(
    reg_meta,
    "xgb_profit_regressor_meta.pkl"
)

print("Semua artifact berhasil disimpan.")

In [ ]:
from xgboost import XGBClassifier

clf_loaded = XGBClassifier()
clf_loaded.load_model("xgb_late_classifier.json")

clf_meta_loaded = joblib.load(
    "xgb_late_classifier_meta.pkl"
)

X_check = np.hstack([
    clf_meta_loaded["encoder"].transform(
        clf_test[clf_meta_loaded["cat_cols"]]
    ),
    clf_test[clf_meta_loaded["num_cols"]].values
])

p_loaded = clf_loaded.predict_proba(X_check)[:, 1]

p_orig = xgb_clf.predict_proba(X_test_c)[:, 1]

print(
    "Classifier identik:",
    np.allclose(p_loaded, p_orig)
)

In [ ]:
from xgboost import XGBRegressor

reg_loaded = XGBRegressor()
reg_loaded.load_model("xgb_profit_regressor.json")

reg_meta_loaded = joblib.load(
    "xgb_profit_regressor_meta.pkl"
)

X_reg_check = reg_meta_loaded["preprocessor"].transform(
    X_test_noTE[reg_meta_loaded["feature_cols"]]
)

p_loaded_reg = reg_loaded.predict(X_reg_check)

p_orig_reg = xgb_model_noTE.predict(
    X_test_noTE[reg_meta_loaded["feature_cols"]]
)

print(
    "Regressor identik:",
    np.allclose(p_loaded_reg, p_orig_reg)
)

In [ ]:
from google.colab import files

files.download("xgb_late_classifier.json")
files.download("xgb_late_classifier_meta.pkl")
files.download("xgb_profit_regressor.json")
files.download("xgb_profit_regressor_meta.pkl")